# HookLink2iOS v4.0 (X3) — IPA do último Actions → Colab → Drive

**Transferência de 64 MiB por requisição HTTP Range e por chunk resumível do Google Drive.**

Cada etapa mostra o número, o andamento, bytes transferidos, velocidade e progresso visual. A extração, os hashes SHA-256/MD5 e os testes CRC da IPA também mostram barras de progresso.

**Correção da RomFS:** o campo `build=` do arquivo interno `romfs.id` pode preservar a procedência do conteúdo. O notebook o confere contra o `build-report.json` auditado, valida `title`, `version`, `size` e `index`, e valida separadamente o commit nxnative X3 e a ABI de shaders. Não exige incorretamente que `build=` tenha o commit X3.

Usa **somente o último workflow v4 bem-sucedido** da branch `HookLink2iOS`, sem fallback para IPA 1.8.5, e só envia a IPA após todas as verificações. Execute as três células de código em ordem; habilite o segredo `GH_TOKEN` no Colab com permissão Actions:read.


In [ ]:
# HookLink2iOS v4 ONLY: verify exact upstream SHA and rebuild chain, 64 MiB per chunk
REPO="joao9vegeto-stack/KintTany"
BRANCH="HookLink2iOS"
TITLE="0400486024DEC000"
PROJECT_DRIVE_FOLDER="1IynIGg57WnKydeqsdVzOp90tPkiX_eNY"
CHUNK=64*1024*1024
assert CHUNK == 64*1024*1024 and CHUNK%(256*1024)==0
import os,time,re,json,io,hashlib,zipfile,plistlib,shutil,getpass
from pathlib import Path
from urllib.parse import urlparse
import requests
from google.colab import auth,userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from tqdm.auto import tqdm
from datetime import datetime
STEP=0
TOTAL_STEPS=10
def stage(text):
    global STEP
    STEP+=1
    print(f"\n{'='*66}\n[{STEP:02d}/{TOTAL_STEPS}] {text}\n{'='*66}",flush=True)
def progress_hash(path,algo="sha256"):
    path=Path(path)
    h=hashlib.new(algo)
    total=path.stat().st_size
    with tqdm(total=total,unit="B",unit_scale=True,unit_divisor=1024,
              desc=f"Hash {algo.upper()}: {path.name[:22]}",dynamic_ncols=True,mininterval=.5) as bar:
        with path.open("rb") as f:
            while True:
                block=f.read(8*1024*1024)
                if not block:break
                h.update(block);bar.update(len(block))
    return h.hexdigest()

import google.auth
ROOT=Path("/content/HookLink2iOS-transfer");ROOT.mkdir(exist_ok=True)
stage("Preparar diretório e transferência de 64 MiB")
print("Diretório:",ROOT,"| Tamanho de bloco:",CHUNK//(1024*1024),"MiB",flush=True)
stage("Autenticar GitHub / localizar Actions")
try:TOKEN=userdata.get("GH_TOKEN") or ""
except Exception:TOKEN=""
if not TOKEN:TOKEN=getpass.getpass("GH_TOKEN (Actions:read para localizar a última IPA): ").strip()
if not TOKEN:raise RuntimeError("GH_TOKEN é obrigatório para localizar artefatos neste repositório")
HEAD={"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
def gitget(path,**opts):
    headers=dict(HEAD)
    if TOKEN:headers["Authorization"]="Bearer "+TOKEN
    r=requests.get("https://api.github.com/repos/"+REPO+path,headers=headers,timeout=40,**opts)
    if r.status_code in (401,403):raise RuntimeError("Configure Colab Secrets GH_TOKEN com permissão Actions read.")
    r.raise_for_status();return r.json()
_HASH_CACHE={}
def sha(path,algo="sha256"):
    p=Path(path)
    stat=p.stat()
    cache_key=(str(p.resolve()),algo,stat.st_size,stat.st_mtime_ns)
    if cache_key not in _HASH_CACHE:
        _HASH_CACHE[cache_key]=progress_hash(p,algo)
    return _HASH_CACHE[cache_key]
stage("Autenticar Google Drive")
auth.authenticate_user()
creds,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive=build("drive","v3",credentials=creds,cache_discovery=False)
stage("Selecionar último run 4.0 realmente concluído")
runs=gitget("/actions/runs",params={"branch":BRANCH,"status":"success","per_page":100})["workflow_runs"]
V4_WORKFLOW=".github/workflows/hooklink2ios-v4.yml"
runs=[r for r in runs if r.get("head_branch")==BRANCH and r.get("path")==V4_WORKFLOW and r.get("conclusion")=="success"]
runs.sort(key=lambda r:(r.get("created_at",""),int(r.get("id",0))),reverse=True)
if not runs:raise RuntimeError("Nenhum build HookLink 4.0 concluído com sucesso nesta branch. Sem fallback para a versão 1.8.5.")
chosenrun=runs[0] # Somente o último run V4 bem-sucedido: não retroceder para um artifact antigo.
opts=gitget(f"/actions/runs/{chosenrun['id']}/artifacts").get("artifacts",[])
opts=[art for art in opts if not art.get("expired") and art.get("name","").startswith("HookLink2iOS-V4-")
      and int(art.get("size_in_bytes",0))>20*1024*1024
      and (art.get("workflow_run") or {}).get("id",chosenrun["id"])==chosenrun["id"]
      and (art.get("workflow_run") or {}).get("head_sha",chosenrun["head_sha"])==chosenrun["head_sha"]]
if len(opts)!=1:raise RuntimeError(f"Último run V4 {chosenrun['id']} não possui um artefato único e íntegro ({len(opts)} encontrados). Não usar build anterior.")
artifact=opts[0]
print(f"Último build válido: run {chosenrun['id']} / #{chosenrun['run_number']} / SHA {chosenrun['head_sha'][:12]}")
print(f"Artifact: {artifact['name']} (ID {artifact['id']}, {artifact['size_in_bytes']:,} bytes)")
print(f"Transferências: blocos HTTP Range e Drive resumable de {CHUNK//(1024*1024)} MiB.")
print("Fonte aprovada: última compilação bem-sucedida do HookLink 4.0.",flush=True)


In [ ]:
# ETAPAS 5–9: transferir archive GitHub com Range 64 MiB, extrair, auditar IPA.
def download_actions(art):
    meta=gitget("/actions/artifacts/"+str(art["id"]))
    if meta.get("expired"):raise RuntimeError("Artefato GitHub expirado.")
    if meta.get("name")!=art["name"] or int((meta.get("workflow_run") or {}).get("id",chosenrun["id"]))!=int(chosenrun["id"]):
        raise RuntimeError("Metadados do artefato não correspondem ao último run da versão 4.0.")
    digest=meta.get("digest") or ""
    if not re.fullmatch(r"sha256:[0-9a-f]{64}",digest):
        raise RuntimeError("Sem SHA-256 confiável para o artefato GitHub.")
    total=int(meta["size_in_bytes"])
    partial=ROOT/f"artifact-{art['id']}.zip.part"
    finished=ROOT/f"artifact-{art['id']}.zip"
    marker=ROOT/f"artifact-{art['id']}.meta.json"
    identity={"id":art["id"],"bytes":total,"digest":digest}
    if finished.exists():
        if finished.stat().st_size==total and sha(finished)==digest.split(":",1)[1]:
            print("Arquivo completo já existente: SHA-256 OK. Download dispensado.",flush=True)
            return finished
        finished.unlink()
    if not marker.exists() or json.loads(marker.read_text())!=identity:
        partial.unlink(missing_ok=True)
        marker.write_text(json.dumps(identity))
    pos=partial.stat().st_size if partial.exists() else 0
    if pos>total:partial.unlink();pos=0
    if pos:print(f"Retomando em {pos/1024**2:.1f} MiB de {total/1024**2:.1f} MiB",flush=True)
    with tqdm(total=total,initial=pos,unit="B",unit_scale=True,unit_divisor=1024,
              desc="GitHub → Colab | 64 MiB",dynamic_ncols=True,mininterval=.5) as bar:
        while pos<total:
            end=min(total,pos+CHUNK)-1
            bar.set_postfix_str(f"chunk {pos//CHUNK+1}/{(total+CHUNK-1)//CHUNK}")
            for attempt in range(8):
                received=0
                try:
                    hdr={**HEAD,"Authorization":"Bearer "+TOKEN}
                    rr=requests.get(f"https://api.github.com/repos/{REPO}/actions/artifacts/{art['id']}/zip",
                                    headers=hdr,allow_redirects=False,timeout=40)
                    if rr.status_code not in (301,302,303,307,308) or not rr.headers.get("Location"):
                        raise RuntimeError(f"GitHub não forneceu URL do artefato: HTTP {rr.status_code}")
                    signed=rr.headers["Location"]
                    if urlparse(signed).scheme!="https":raise RuntimeError("URL de download não usa HTTPS")
                    with requests.get(signed,headers={"Range":f"bytes={pos}-{end}"},
                                      stream=True,timeout=(30,180)) as resp:
                        m=re.fullmatch(r"bytes\s+(\d+)-(\d+)/(\d+)",resp.headers.get("Content-Range",""))
                        if resp.status_code!=206 or not m or tuple(map(int,m.groups()))!=(pos,end,total):
                            raise RuntimeError(f"Range inválido: HTTP {resp.status_code}, Content-Range={resp.headers.get('Content-Range')!r}")
                        with partial.open("r+b" if partial.exists() else "w+b") as f:
                            f.truncate(pos);f.seek(pos)
                            for block in resp.iter_content(1024*1024):
                                if block:
                                    f.write(block);received+=len(block);bar.update(len(block))
                        if received!=end-pos+1:raise RuntimeError(f"Chunk incompleto: {received} / {end-pos+1} bytes")
                    pos=end+1
                    break
                except Exception as exc:
                    bar.update(-received)
                    if partial.exists():
                        with partial.open("r+b") as f:f.truncate(pos)
                    tqdm.write(f"Chunk {pos//CHUNK+1}: tentativa {attempt+1}/8 falhou: {type(exc).__name__}: {exc}")
                    if attempt==7:raise
                    time.sleep(min(30,2**attempt))
    partial.replace(finished)
    return finished

stage("Baixar artefato GitHub em chunks HTTP Range de 64 MiB")
SOURCE=download_actions(artifact)
origin=f"Run {chosenrun['id']} / artefato {artifact['id']}"
stage("Verificar SHA-256 completo do arquivo recebido")
expected_digest=(artifact.get("digest") or gitget("/actions/artifacts/"+str(artifact["id"])).get("digest") or "").split(":",1)[-1]
assert SOURCE.stat().st_size==int(artifact["size_in_bytes"]),"Tamanho do artefato ZIP divergente"
if sha(SOURCE)!=expected_digest:raise RuntimeError("SHA-256 do artefato GitHub não confere")
print("GitHub ZIP: SHA-256 OK",flush=True)

stage("Inspecionar artefato e extrair IPA com progresso contínuo")
with zipfile.ZipFile(SOURCE) as z:
    nested=[info for info in z.infolist() if info.filename.lower().endswith(".ipa") and not info.is_dir()]
    reports=[info for info in z.infolist() if Path(info.filename).name=="build-report.json"]
    if len(nested)!=1 or len(reports)!=1:
        raise RuntimeError(f"Artefato v4 inválido: IPAs={len(nested)}, relatórios={len(reports)}")
    info=nested[0]
    report=json.loads(z.read(reports[0]))
    if report.get("hooklink_release")!="4.0":raise RuntimeError("Relatório não é HookLink 4.0")
    ENGINE="ee3ce6a02bc737d38795f0a96cce494cee7a799c"
    if report.get("engine_commit")!=ENGINE:raise RuntimeError("Motor nxnative X3 divergente")
    if report.get("shader_chain")!="b6112b954691":raise RuntimeError("Shader ABI X3 divergente")
    if report.get("original_1_8_shader_image_reused") is not False:raise RuntimeError("Shaders da 1.8.5 foram reutilizados")
    if report.get("romfs_mode")!="separate" or report.get("gpu_arch")!="none":raise RuntimeError("Não é build thin / GPU none")
    if str(report.get("run_id"))!=str(chosenrun["id"]):raise RuntimeError("Relatório pertence a outro run")
    if report.get("commit")!=chosenrun["head_sha"]:raise RuntimeError("Commit no relatório não corresponde ao run")
    IPA=ROOT/Path(info.filename).name
    wanted=report.get("ipa",{}).get("sha256")
    if not re.fullmatch("[0-9a-f]{64}",str(wanted)):raise RuntimeError("Relatório sem SHA-256 da IPA")
    if int(report["ipa"].get("size",0))!=info.file_size:raise RuntimeError("IPA diverge do tamanho declarado pelo build")
    if IPA.exists() and IPA.stat().st_size==info.file_size and sha(IPA)==wanted:
        print("IPA já extraída e validada; extração dispensada.",flush=True)
    else:
        tmp=IPA.with_suffix(".ipa.part")
        with z.open(info) as src,tmp.open("wb") as dst:
            with tqdm(total=info.file_size,unit="B",unit_scale=True,unit_divisor=1024,
                      desc="Extração da IPA",dynamic_ncols=True,mininterval=.5) as bar:
                while True:
                    block=src.read(8*1024*1024)
                    if not block:break
                    dst.write(block);bar.update(len(block))
        tmp.replace(IPA)
stage("Validar SHA-256, motor X3, RomFS, bibliotecas Metal e arquivos ZIP")
if IPA.stat().st_size>=4_000_000_000:raise RuntimeError("IPA excede o limite seguro de 4 GB")
IPA_HASH=sha(IPA)
if IPA_HASH!=report["ipa"]["sha256"]:raise RuntimeError("SHA-256 IPA divergente do build-report.json")
with zipfile.ZipFile(IPA) as z:
    names=z.namelist()
    pre="Payload/DmIOS.app/"
    if len(names)!=len(set(names)) or any(pre+key not in names for key in ("DmIOS","romfs.id","Info.plist")):
        raise RuntimeError("IPA incompleta ou contém entradas duplicadas")
    embedded=z.read(pre+"romfs.id").decode("utf8","replace").strip()
    declared=str(report.get("ipa",{}).get("contents",{}).get("romfs_marker","")).strip()
    if not declared or embedded!=declared:
        raise RuntimeError(f"romfs.id não coincide com o marcador auditado no build-report. IPA: {embedded!r} | relatório: {declared!r}")
    target=str(report.get("romfs",{}).get("identity","")).strip()
    if not embedded.startswith("DMROMFSID1 ") or not target.startswith("DMROMFSID1 "):
        raise RuntimeError("Formato de identidade RomFS não reconhecido")
    def fields(value):
        return dict(part.split("=",1) for part in value.split()[1:] if "=" in part)
    actual=fields(embedded);expected=fields(target)
    for key in ("title","version","size","index"):
        if not expected.get(key) or actual.get(key)!=expected[key]:
            raise RuntimeError(f"RomFS {key} divergente: IPA={actual.get(key)!r}, build={expected.get(key)!r}")
    if actual["title"]!=TITLE or actual["version"]!="v131072":
        raise RuntimeError("Title ID ou versão da RomFS não correspondem ao jogo")
    if target.find("build="+ENGINE)==-1:raise RuntimeError("Relatório da RomFS não referencia o motor X3")
    if actual.get("build")!=ENGINE:
        print("AVISO: romfs.id preserva marcador de origem:",actual.get("build"),
              "| motor X3 verificado separadamente por SHA/relatório.",flush=True)
    plist=plistlib.loads(z.read(pre+"Info.plist"))
    if str(plist.get("MinimumOSVersion"))!="26.0":raise RuntimeError("IPA não está preparada para iOS 26")
    if not any(re.fullmatch(r"Payload/DmIOS\.app/msl/b-[^/]+\.img(?:\.[0-9]+)?",entry) for entry in names):
        raise RuntimeError("Bibliotecas Metal ausentes")
    if any(entry.lower().endswith("romfs.img") for entry in names):
        raise RuntimeError("RomFS deveria estar fora da IPA")
    if any(entry.startswith(pre+"pso/") and entry.endswith(".mtl4a") for entry in names):
        raise RuntimeError("GPU none: IPA contém PSO offline")
    infos=[info for info in z.infolist() if not info.is_dir()]
    total_uncompressed=sum(info.file_size for info in infos)
    with tqdm(total=total_uncompressed,unit="B",unit_scale=True,unit_divisor=1024,
              desc="CRC de todos os arquivos",dynamic_ncols=True,mininterval=.5) as bar:
        for info in infos:
            bar.set_postfix_str(Path(info.filename).name[:28])
            with z.open(info) as f:
                while True:
                    block=f.read(8*1024*1024)
                    if not block:break
                    bar.update(len(block))
print("VALIDAÇÃO CONCLUÍDA: versão 4.0 / X3, IPA intacta, RomFS externa.",flush=True)
print("Arquivo:",IPA,"| tamanho:",IPA.stat().st_size,"| SHA-256:",IPA_HASH,flush=True)


In [ ]:
# ETAPA 10: upload Google Drive resumable 64 MiB e verificação de integridade.
stage("Preparar pasta HookLink2iOS/ipa e upload resumível de 64 MiB")
def drive_folder():
    parent=PROJECT_DRIVE_FOLDER
    try:
        drive.files().get(fileId=parent,fields="id",supportsAllDrives=True).execute()
        print("Pasta do projeto encontrada no Drive.",flush=True)
    except Exception:
        matches=drive.files().list(q="name='HookLink2iOS' and mimeType='application/vnd.google-apps.folder' and trashed=false",
            fields="files(id)",pageSize=20).execute().get("files",[])
        parent=matches[0]["id"] if matches else drive.files().create(body={
            "name":"HookLink2iOS","mimeType":"application/vnd.google-apps.folder"},fields="id").execute()["id"]
    entries=drive.files().list(q=f"'{parent}' in parents and name='ipa' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields="files(id)",pageSize=20).execute().get("files",[])
    fid=entries[0]["id"] if entries else drive.files().create(body={
        "name":"ipa","mimeType":"application/vnd.google-apps.folder","parents":[parent]},fields="id").execute()["id"]
    print("Destino Drive /HookLink2iOS/ipa:",fid,flush=True)
    return fid
folder=drive_folder()
name=IPA.stem+"-"+IPA_HASH[:12]+".ipa"
queryname=name.replace("'","\\'")
existing=drive.files().list(q=f"'{folder}' in parents and name='{queryname}' and trashed=false",
    fields="files(id,name,size,md5Checksum,appProperties,webViewLink)",pageSize=100).execute().get("files",[])
print("Conferindo MD5 local, em progresso...",flush=True)
MD5=sha(IPA,"md5")
current=next((x for x in existing if int(x.get("size",0))==IPA.stat().st_size
    and x.get("md5Checksum")==MD5 and (x.get("appProperties") or {}).get("sha256")==IPA_HASH),None)
if current:
    print("IPA idêntica já no Drive. Upload dispensado.",flush=True)
else:
    upload=MediaFileUpload(str(IPA),mimetype="application/octet-stream",chunksize=CHUNK,resumable=True)
    request=drive.files().create(body={"name":name,"parents":[folder],
        "appProperties":{"sha256":IPA_HASH,"titleId":TITLE,"branch":BRANCH,"engine":ENGINE,"hooklink":"4.0"}},
        media_body=upload,fields="id,name,size,md5Checksum,appProperties,webViewLink")
    current=None;uploaded=0;total=IPA.stat().st_size
    with tqdm(total=total,unit="B",unit_scale=True,unit_divisor=1024,
              desc="Colab → Drive | 64 MiB",dynamic_ncols=True,mininterval=.5) as bar:
        while current is None:
            for attempt in range(8):
                try:
                    status,current=request.next_chunk()
                    pos=total if current is not None else int(status.resumable_progress) if status else uploaded
                    if pos>uploaded:bar.update(pos-uploaded)
                    uploaded=max(uploaded,pos)
                    bar.set_postfix_str(f"chunk {(uploaded+CHUNK-1)//CHUNK}/{(total+CHUNK-1)//CHUNK}")
                    break
                except Exception as exc:
                    tqdm.write(f"Upload: tentativa {attempt+1}/8 falhou: {type(exc).__name__}: {exc}")
                    if attempt==7:raise
                    time.sleep(min(30,2**attempt))
stage("Conferir upload: tamanho, MD5 e SHA-256 do Drive")
print("Verificando tamanho, MD5 e SHA-256 no Drive...",flush=True)
check=drive.files().get(fileId=current["id"],fields="id,name,size,md5Checksum,appProperties,webViewLink",
    supportsAllDrives=True).execute()
if int(check.get("size",0))!=IPA.stat().st_size or check.get("md5Checksum")!=MD5 or (check.get("appProperties") or {}).get("sha256")!=IPA_HASH:
    raise RuntimeError("Falha de readback Drive: tamanho/MD5/SHA-256 divergentes")
print("TRANSFERÊNCIA FINALIZADA E VALIDADA.",flush=True)
print("Origem:",origin,"| Nome:",check["name"],flush=True)
print("SHA-256:",IPA_HASH,flush=True)
print("Drive:",check.get("webViewLink") or "https://drive.google.com/file/d/"+check["id"]+"/view",flush=True)
